# S43 오차 분석: 생성 기준 v16의 손실 몫과 유형

판정 세트 330질의는 라벨과 건수만, 세트 밖 100질의는 서브에이전트 판독 표를 집계한다(`docs/slices/43-error-analysis.md`, D-76 결정 6). 저장된 기록만 읽고 API를 부르지 않는다. 답변·질의·judge 이유 원문은 출력하지 않는다.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

In [2]:
ROOT = Path.cwd().parent
JUDGE = ROOT / "data/gen_eval/judge"
STEM = "hybrid-cond-filter-law-ms256-k20-ce-v16-dev-full-spk-marks{s}-deepseek_go_mixed-j_v13.jsonl"
FULL = JUDGE / STEM.format(s="")
OUTSET = JUDGE / STEM.format(s="-outset100")
TABLE = ROOT / "data/gen_eval/error_analysis/s43-outset100.csv"
WRONG = ["근거 없이 답함", "안내 없이 부분 답함", "무언 생략", "누락", "상황 미전달", "과잉 거부", "불필요한 부분 안내"]

In [3]:
def load(path):
    with path.open(encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def scores(r):
    return (r.get("faithfulness") or {}).get("score"), (r.get("relevance") or {}).get("score")


def loss(r):
    f, g = scores(r)
    return None if f is None or g is None else 1 - (f + g) / 2


def evidence_group(r):
    # rag.gen_eval.report.refusal_table의 묶음과 같은 정의다
    if not r["gold_n"]:
        return "정답 문서 없음"
    if r["answer_in_chunk"]:
        return "정답 문서 있음, 청크에 핵심 답 있음"
    if r["answer_in_chunk"] is False:
        return "정답 문서 있음, 청크에 핵심 답 없음"
    return "정답 문서 있음, 판단 불가"


def share_table(rows, key):
    kept = [r for r in rows if loss(r) is not None]
    total = sum(loss(r) for r in kept)
    table = {}
    for r in kept:
        row = table.setdefault(key(r), {"질의": 0, "손실 있는 질의": 0, "손실 합": 0.0})
        row["질의"] += 1
        row["손실 있는 질의"] += loss(r) > 0
        row["손실 합"] += loss(r)
    out = pd.DataFrame(table).T
    assert abs(out["손실 합"].sum() - total) < 1e-9
    out["손실 몫"] = (out["손실 합"] / total).round(3)
    out["손실 합"] = out["손실 합"].round(2)
    return out.astype({"질의": int, "손실 있는 질의": int}).sort_values("손실 합", ascending=False)

## 판정 세트 손실 몫

라벨과 건수만 쓴다. 종합 손실은 `1 − (충실성 + 관련성) / 2`이고 두 값이 모두 있는 질의만 쓴다.

In [4]:
full = list(load(FULL).values())
kept = [r for r in full if loss(r) is not None]
total = sum(loss(r) for r in kept)
print(f"질의 {len(full)}, 계산 {len(kept)}, 손실 합 {total:.2f}, 평균 종합 {1 - total / len(kept):.4f}")
share_table(full, evidence_group)

질의 330, 계산 325, 손실 합 21.39, 평균 종합 0.9342


,질의,손실 있는 질의,손실 합,손실 몫
"정답 문서 있음, 청크에 핵심 답 있음",237,50,12.49,0.584
"정답 문서 있음, 판단 불가",50,21,5.33,0.249
"정답 문서 있음, 청크에 핵심 답 없음",27,13,3.07,0.143
정답 문서 없음,11,2,0.50,0.023


In [5]:
share_table(full, lambda r: (r.get("relevance") or {}).get("label"))

,질의,손실 있는 질의,손실 합,손실 몫
일부,73,73,18.62,0.87
답하지 않음,3,3,1.50,0.07
답함,249,10,1.28,0.06


In [6]:
share_table(full, lambda r: "충실성 1 미만" if scores(r)[0] < 1 else "충실성 1")

,질의,손실 있는 질의,손실 합,손실 몫
충실성 1,312,73,19.00,0.888
충실성 1 미만,13,13,2.39,0.112


In [7]:
Counter(i["outcome"] for r in full for i in r["item_eval"]["items"]).most_common()

[('맞음', 250),
 ('근거 없이 답함', 59),
 ('안내 없이 부분 답함', 51),
 ('상황 전달', 42),
 ('상황 미전달', 33),
 ('무언 생략', 27),
 ('누락', 15),
 ('과잉 거부', 2)]

## 프롬프트 지시와 judge 규칙의 어긋남: 판정 세트 라벨

항목 상태 "없음"에 답변이 무언가를 전한 경우다. 항목 처리는 참고값이고(I11) 종합 손실은 충실성과 관련성에서만 온다. 상황을 밝힌 처리는 판정표 I8에서 "답함과 같음"이다.

In [8]:
def absent_handling(r):
    hs = {i["handling"] for i in r["item_eval"]["items"] if i["state"] == "없음"}
    if "상황을 밝히고 답함" in hs:
        return "없음 항목, 상황을 밝히고 답함 포함"
    if hs & {"답함", "부분임을 밝히고 답함"}:
        return "없음 항목, 답함"
    if hs:
        return "없음 항목, 거부·다루지 않음만"
    return "없음 항목 없음"


pairs = Counter((i["state"], i["handling"]) for r in full for i in r["item_eval"]["items"] if i["state"] == "없음")
print(pairs.most_common())
share_table(full, absent_handling)

[(('없음', '답함'), 42), (('없음', '다루지 않음'), 27), (('없음', '거부'), 22), (('없음', '상황을 밝히고 답함'), 16), (('없음', '부분임을 밝히고 답함'), 1)]


,질의,손실 있는 질의,손실 합,손실 몫
없음 항목 없음,244,53,13.22,0.618
"없음 항목, 거부·다루지 않음만",33,21,5.50,0.257
"없음 항목, 답함",34,8,1.88,0.088
"없음 항목, 상황을 밝히고 답함 포함",14,4,0.79,0.037


In [9]:
rows = []
for name in ("없음 항목, 상황을 밝히고 답함 포함", "없음 항목, 답함"):
    labels = Counter((r.get("relevance") or {}).get("label") for r in kept if absent_handling(r) == name)
    rows.append({"묶음": name, **labels})
pd.DataFrame(rows).fillna(0)

,묶음,답함,일부
0,"없음 항목, 상황을 밝히고 답함 포함",12,2
1,"없음 항목, 답함",28,6


## 세트 밖 100질의: 판독 표

대상은 손실이나 틀린 처리가 있는 47질의다. `type`은 판독자, `where_to_fix`는 반박 판독자의 판단이고(judge 오판 의심과 어긋남 26질의), 사람 검수를 거치지 않았다.

In [10]:
outset = load(OUTSET)
table = pd.read_csv(TABLE, encoding="utf-8").fillna("")
assert set(table.qid) == {q for q, r in outset.items() if (loss(r) is None or loss(r) > 0) or any(i["outcome"] in WRONG for i in r["item_eval"]["items"])}
print("대상", len(table), "손실 합", round(sum(loss(r) for r in outset.values() if loss(r) is not None), 2))
recheck = table[table.recheck_type != ""]
print("재판독 일치: 유형", int((recheck.type == recheck.recheck_type).sum()), "/", len(recheck), ", 지시·judge 관계", int((recheck.prompt_judge == recheck.recheck_prompt_judge).sum()), "/", len(recheck))
print("지어낸 내용", int(table.fabricated.astype(str).eq("True").sum()), "질의, 반박 판독:", table.challenge_verdict[table.challenge_verdict != ""].value_counts().to_dict())

대상 47 손실 합 4.77
재판독 일치: 유형 6 / 8 , 지시·judge 관계 5 / 8
지어낸 내용 4 질의, 반박 판독: {'judge 오판 맞음': 10, '어긋남 맞음': 6, '갈림': 5, '어긋남 아님': 4, 'judge 판정 맞음': 1}


In [11]:
info = table.set_index("qid")
n_kept = sum(loss(r) is not None for r in outset.values())
rows = list(outset.values())


def reader_type(r):
    return info.type.get(r["qid"], "손실·틀린 처리 없음")


t = share_table(rows, reader_type)
t["상한(종합 지표)"] = (t["손실 합"] / n_kept).round(4)
t

,질의,손실 있는 질의,손실 합,손실 몫,상한(종합 지표)
관련 발언으로 대신함,7,3,1.00,0.210,0.0102
judge 오판 의심,11,4,1.00,0.210,0.0102
질의자 말 전달,11,3,0.75,0.157,0.0077
항목 누락,1,1,0.50,0.105,0.0051
상황 전달,4,2,0.50,0.105,0.0051
부분 답 안내 없음,6,1,0.42,0.087,0.0043
사실·발언자 오류,3,2,0.35,0.073,0.0036
질의 문제,2,1,0.25,0.052,0.0026
손실·틀린 처리 없음,53,0,0.00,0.000,0.0000


In [12]:
def cause(r):
    # 반박 판독이 있으면 그 판단을, 없으면 판독 유형을 쓴다
    q = r["qid"]
    if q not in info.index:
        return "손실·틀린 처리 없음"
    if info.where_to_fix[q]:
        return info.where_to_fix[q]
    return {"질의 문제": "질의 문제", "검색 실패": "검색 실패"}.get(info.type[q], "답변 오류")


t = share_table(rows, cause)
t["상한(종합 지표)"] = (t["손실 합"] / n_kept).round(4)
t

,질의,손실 있는 질의,손실 합,손실 몫,상한(종합 지표)
답변 오류,19,7,2.27,0.476,0.0232
judge 적용 잡음,14,6,1.50,0.315,0.0153
규칙과 지시의 충돌,9,2,0.50,0.105,0.0051
질의 문제,2,1,0.25,0.052,0.0026
생성 지시 부족,1,1,0.25,0.052,0.0026
손실·틀린 처리 없음,53,0,0.00,0.000,0.0000


## 질의자 말 전달의 관련성 라벨

규칙 R8은 질의자의 말만 전하고 답변자의 입장이 없으면 일부로 정한다. 같은 유형의 판정이 일관된지 본다.

In [13]:
mask = table.type == "질의자 말 전달"
labels = Counter((outset[q].get("relevance") or {}).get("label") for q in table.qid[mask])
conflict = table.where_to_fix == "규칙과 지시의 충돌"
print("질의자 말 전달", int(mask.sum()), dict(labels))
print("규칙과 지시의 충돌", int(conflict.sum()), dict(Counter((outset[q].get("relevance") or {}).get("label") for q in table.qid[conflict])))

질의자 말 전달 11 {'답함': 8, '일부': 3}
규칙과 지시의 충돌 9 {'답함': 7, '일부': 2}


## 두 집합 비교

세트 밖 비율이 판정 세트와 크게 다르면 상한을 판정 세트에 옮겨 읽을 수 없다.

In [14]:
rows2 = []
for name, data in (("판정 세트", full), ("세트 밖", list(outset.values()))):
    ok = [r for r in data if loss(r) is not None]
    wrong = sum(i["outcome"] in WRONG for r in data for i in r["item_eval"]["items"])
    partial = sum((r.get("relevance") or {}).get("label") == "일부" for r in ok)
    rows2.append({"집합": name, "질의": len(data), "평균 종합": round(1 - sum(map(loss, ok)) / len(ok), 4), "관련성 일부 비율": round(partial / len(ok), 3), "틀린 처리 항목/100질의": round(wrong / len(data) * 100, 1)})
pd.DataFrame(rows2)

,집합,질의,평균 종합,관련성 일부 비율,틀린 처리 항목/100질의
0,판정 세트,330,0.9342,0.225,56.7
1,세트 밖,100,0.9514,0.122,53.0
